# 02 — Preparación de datos en Colab Enterprise

Este notebook prepara el dataset para entrenar usando **Google Cloud Storage** (`gs://jbj-vision`):
1. Autentica en GCP
2. Sincroniza el repo desde GCS al runtime
3. Instala dependencias
4. Descarga imágenes desde los CSV (cache local en `/content/cache`)
5. Persiste el cache como tarball en GCS para no re-descargar en próximas sesiones
6. Genera splits estratificados train/val/test en GCS

> **Antes de ejecutar:** subí el repo al bucket con:
> ```bash
> gsutil -m rsync -r -x '\.git/.*|__pycache__/.*|\.DS_Store' \
>   /ruta/local/UdeSA-tp-vision-jbj/ gs://jbj-vision/code/
> ```


## 1. Auth + repo en el runtime

In [1]:
from google.colab import auth
auth.authenticate_user()
print('Autenticación exitosa.')

BUCKET   = 'gs://jbj-vision'
REPO_DIR = '/content/repo'
CACHE_ROOT = '/content/cache'

# Sincronizar repo desde GCS
!gsutil -m rsync -r -x '\.git/.*|__pycache__/.*' {BUCKET}/code/ {REPO_DIR}
%cd {REPO_DIR}
!ls


NotImplementedError: google.colab.drive.mount is not supported in Colab Enterprise.

## 2. Instalar dependencias

In [ ]:
!pip install -q gcsfs==2023.10.0 fsspec google-cloud-storage
!pip install -q -r requirements.txt


## 3. Setup paths con GCS

In [ ]:
import sys
sys.path.insert(0, '/content/repo')

from src.data.colab import setup_gcp
config = setup_gcp('config/pipeline_config.yaml', bucket='gs://jbj-vision', local_cache_root='/content/cache')
print('CSVs:')
print('  pants:', config['data']['pants_csv'])
print('  tops: ', config['data']['tops_csv'])
print('Cache imagenes:')
print('  pants:', config['paths']['images_pants'])
print('  tops: ', config['paths']['images_tops'])
print('Splits GCS:', config['paths']['splits'])
print('Modelos GCS:', config['paths']['models'])


## 4. Pull cache de imágenes desde GCS (si existe)

Si ya corriste este notebook antes, el cache de imágenes está en GCS como tarball.
Lo descargamos para no re-bajar las ~30k imágenes desde internet.

In [ ]:
import subprocess, os

def pull_tarball(ds: str):
    """Descarga y extrae el tarball de imágenes desde GCS si existe."""
    gcs_path = f"gs://jbj-vision/data/processed/images_{ds}.tar.gz"
    local_tar = f"/tmp/images_{ds}.tar.gz"
    r = subprocess.run(['gsutil', 'ls', gcs_path], capture_output=True)
    if r.returncode == 0:
        print(f'Pulling {gcs_path} ...')
        os.makedirs(config['paths'][f'images_{ds}'], exist_ok=True)
        !gsutil -m cp {gcs_path} {local_tar}
        !tar xzf {local_tar} -C {config['paths'][f'images_{ds}']} --strip-components=1
        print(f'✓ Cache {ds} restaurado.')
        return True
    else:
        print(f'No existe tarball para {ds}, se descargará desde internet.')
        return False

pants_cache_ok = pull_tarball('pants')
tops_cache_ok  = pull_tarball('tops')


## 5. Descargar imágenes faltantes

Descarga ~15k imágenes de cada CSV. Idempotente: saltea las que ya están en cache.
Si el tarball se restauró arriba, esta celda es muy rápida (solo verifica el cache).


In [ ]:
SAMPLE_PANTS = 15000
SAMPLE_TOPS  = 15000

from src.data.downloader import download_csv_images

log_pants = download_csv_images(
    csv_path=config['data']['pants_csv'],
    cache_dir=config['paths']['images_pants'],
    sample=SAMPLE_PANTS,
    workers=8,
    timeout=15,
    log_path=f"{config['paths']['splits']}/pants_download_log.csv",
)
print(f"\nPants OK: {log_pants['success'].sum()}/{len(log_pants)}")


In [ ]:
log_tops = download_csv_images(
    csv_path=config['data']['tops_csv'],
    cache_dir=config['paths']['images_tops'],
    sample=SAMPLE_TOPS,
    workers=8,
    timeout=15,
    log_path=f"{config['paths']['splits']}/tops_download_log.csv",
)
print(f"\nTops OK: {log_tops['success'].sum()}/{len(log_tops)}")


## 6. Persistir cache en GCS (tarball)

Empaquetamos las imágenes descargadas y las subimos a GCS para no repetir la descarga
en sesiones futuras. Solo sube lo que cambió (incremental).


In [ ]:
def push_tarball(ds: str):
    cache_dir = config['paths'][f'images_{ds}']
    gcs_path  = f"gs://jbj-vision/data/processed/images_{ds}.tar.gz"
    local_tar = f"/tmp/images_{ds}.tar.gz"
    print(f'Empaquetando {cache_dir} ...')
    !tar czf {local_tar} -C {cache_dir} .
    print(f'Subiendo a {gcs_path} ...')
    !gsutil -m cp {local_tar} {gcs_path}
    print(f'✓ Tarball {ds} subido.')

push_tarball('pants')
push_tarball('tops')


## 7. Splits estratificados (filtrando por imágenes descargadas)

In [ ]:
from src.data.splits import generate_splits

# Pants
pants_urls = set(log_pants[log_pants['success']]['url'])
pants_paths = generate_splits(
    csv_path=config['data']['pants_csv'],
    output_dir=config['paths']['splits'],
    stratify_col=config['data']['splits']['stratify_col'],
    train_ratio=config['data']['splits']['train_ratio'],
    val_ratio=config['data']['splits']['val_ratio'],
    test_ratio=config['data']['splits']['test_ratio'],
    min_samples_per_class=config['data']['splits']['min_samples_per_class'],
    seed=config['seed'],
    filter_urls=pants_urls,
)
print('Splits pants:', pants_paths)


In [ ]:
# Tops
tops_urls = set(log_tops[log_tops['success']]['url'])
tops_paths = generate_splits(
    csv_path=config['data']['tops_csv'],
    output_dir=config['paths']['splits'],
    stratify_col=config['data']['splits']['stratify_col'],
    train_ratio=config['data']['splits']['train_ratio'],
    val_ratio=config['data']['splits']['val_ratio'],
    test_ratio=config['data']['splits']['test_ratio'],
    min_samples_per_class=config['data']['splits']['min_samples_per_class'],
    seed=config['seed'],
    filter_urls=tops_urls,
)
print('Splits tops:', tops_paths)


## 8. Sanity check final

In [ ]:
import pandas as pd

BUCKET_SPLITS = config['paths']['splits']
print('=== Resumen ===')
for ds in ['pants_1', 'tops_1']:
    for split in ['train', 'val', 'test']:
        p = f"{BUCKET_SPLITS}/{ds}_{split}.csv"
        try:
            df = pd.read_csv(p)
            print(f'  {ds}/{split}: {len(df):,} filas')
        except Exception as e:
            print(f'  {ds}/{split}: ERROR — {e}')

print('\n✓ Listo para entrenar. Ejecuta 03_train_pants_colab.ipynb')
